# Model Training, Evaluation & SHAP Analysis

Trains five regressors (XGBoost, ExtraTrees, RandomForest, GradientBoosting, SVR)
on the step-2 filtered features for both ΔEST and kRISC targets.

All logic lives in `models.py` and `visualization.py`; this notebook only orchestrates calls.

In [ ]:
%matplotlib inline
import numpy as np

import models as m
import visualization as viz
from config import (
    EST_MEAN, EST_STD, KRISC_MEAN, KRISC_STD,
    EST_OUTLIER_CAP, BAYES_N_ITER,
)

## 1. Load data

In [ ]:
data = m.load_data()
print(f"X_traindev shape: {data['X_traindev'].shape}")
print(f"X_test shape:     {data['X_test_all'].shape}")

## 2. Train ΔEST models

In [ ]:
# Filter missing and cap outliers
X_tr_est, y_tr_est = m.filter_valid_samples(
    data['X_traindev'], data['y_traindev_est'])
X_te_est, y_te_est = m.filter_valid_samples(
    data['X_test_all'], data['y_test_est'])

# Remove EST > outlier cap (in original scale)
tr_real = y_tr_est * EST_STD + EST_MEAN
X_tr_est = X_tr_est[tr_real <= EST_OUTLIER_CAP]
y_tr_est = y_tr_est[tr_real <= EST_OUTLIER_CAP]

te_real = y_te_est * EST_STD + EST_MEAN
X_te_est = X_te_est[te_real <= EST_OUTLIER_CAP]
y_te_est = y_te_est[te_real <= EST_OUTLIER_CAP]

print(f"ΔEST 有效样本: Train={len(y_tr_est)}, Test={len(y_te_est)}")

In [ ]:
est_results = m.train_and_evaluate(
    X_tr_est, y_tr_est,
    X_te_est, y_te_est,
    y_mean=EST_MEAN, y_std=EST_STD,
    target_name='ΔEST',
    feature_names=data['feat_names_est'],
    n_iter=BAYES_N_ITER,
)

## 3. Train kRISC models

In [ ]:
X_tr_krisc, y_tr_krisc = m.filter_valid_samples(
    data['X_traindev'], data['y_traindev_krisc'])
X_te_krisc, y_te_krisc = m.filter_valid_samples(
    data['X_test_all'], data['y_test_krisc'])

print(f"kRISC 有效样本: Train={len(y_tr_krisc)}, Test={len(y_te_krisc)}")

In [ ]:
krisc_results = m.train_and_evaluate(
    X_tr_krisc, y_tr_krisc,
    X_te_krisc, y_te_krisc,
    y_mean=KRISC_MEAN, y_std=KRISC_STD,
    target_name='kRISC',
    feature_names=data['feat_names_krisc'],
    n_iter=BAYES_N_ITER,
)

## 4. Summary table

In [ ]:
all_results = {'ΔEST': est_results, 'kRISC': krisc_results}
m.print_summary_table(all_results)

## 5. Scatter plots

In [ ]:
viz.plot_prediction_scatter(est_results,   'ΔEST')
viz.plot_prediction_scatter(krisc_results, 'kRISC')

## 6. Metrics comparison

In [ ]:
viz.plot_metrics_comparison(est_results,   'ΔEST')
viz.plot_metrics_comparison(krisc_results, 'kRISC')

## 7. Prediction curves

In [ ]:
viz.plot_prediction_curve(
    est_results,   'ΔEST',  model_name='ExtraTrees',
    unit='eV', save_prefix='pred_curve',
)
viz.plot_prediction_curve(
    krisc_results, 'kRISC', model_name='XGBoost',
    unit='ns⁻¹', save_prefix='pred_curve',
)

## 8. Pearson correlation heat-maps

In [ ]:
viz.plot_correlation_matrix(
    X_tr_est, y_tr_est,
    data['feat_names_est'],
    target_name='ΔEST', target_label='ΔEST',
    top_n=15, save_path='pearson_correlation_heatmap_EST.png',
)
viz.plot_correlation_matrix(
    X_tr_krisc, y_tr_krisc,
    data['feat_names_krisc'],
    target_name='kRISC', target_label='kRISC',
    top_n=15, save_path='pearson_correlation_heatmap_KRISC.png',
)

## 9. SHAP analysis

In [ ]:
shap_est, desc_names_est = viz.plot_shap_analysis(
    est_results, X_tr_est,
    data['feat_names_est'],
    target_name='ΔEST',
    model_name='XGBoost',
    save_prefix='shap',
)

In [ ]:
shap_krisc, desc_names_krisc = viz.plot_shap_analysis(
    krisc_results, X_tr_krisc,
    data['feat_names_krisc'],
    target_name='kRISC',
    model_name='XGBoost',
    save_prefix='shap',
)